## Packages Import


In [7]:
import os
import re
import yaml
import requests
import pandas as pd
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup

## Apollo Scraper

In [8]:
with open("config.yaml", "r", encoding="UTF-8") as yf:
    config = yaml.safe_load(yf)
username = config['credentials']['user']
password = config['credentials']['password']
credentials = HTTPBasicAuth(username, password)

In [9]:
group_id = input("Enter group ID: ")
url = f"https://planzajec.uek.krakow.pl/index.php?typ=G&id={group_id}&okres=1"
response = requests.get(url, auth=credentials)
response.encoding = "UTF-8"
print(response.status_code)


200


In [10]:
print(response.status_code)
print(response.text[:500])

200
<!DOCTYPE html PUBLIC "-//W3C//DTD HTML 4.0 Transitional//EN" "http://www.w3.org/TR/REC-html40/loose.dtd">
<html>
<head>
<meta http-equiv="Content-Type" content="text/html; charset=UTF-8">
<title>Plan zajęć UEK ZICSS1-1211</title>
<link rel="stylesheet" type="text/css" href="planzajec.css">
</head>
<body>
<script src="js/accessibility_uek.js"></script><script src="js/apollo2calendar.js"></script><div class="naglowek">
<div class="logo"><img src="UEK-logo.gif" alt=""></div>
<div class="planzajec"


In [11]:
page_dom = BeautifulSoup(response.text, 'html.parser')


In [12]:
group = page_dom.select_one("div.grupa")
if group:
    group = group.get_text(strip=True)
else:
    group = "unknown"
print(group)

ZICSS1-1211


### TABLE DATA FRAME

In [13]:
classes_tag = page_dom.select_one("table")
with open("temp.html", "w", encoding="UTF-8") as hf:
    hf.write(classes_tag.prettify())
classes = pd.read_html("temp.html", encoding="UTF-8")[0]
os.remove("temp.html")

### FILTER

In [14]:
classes = classes.loc[classes['Typ'].isin(["ćwiczenia", "wykład", "egzamin"])]

### SPLIT DAY TIME

In [15]:
split_cols = classes['Dzień, godzina'].str.split(' ', expand=True)
print(split_cols.shape)
print(split_cols.head())

(17, 5)
     0      1  2      3      4
1   Pn  13:15  -  15:45  (3g.)
4   Wt  09:45  -  11:15  (2g.)
8   Wt  15:00  -  16:30  (2g.)
9   Wt  18:30  -  20:00  (2g.)
11  Śr  13:15  -  14:45  (2g.)


### CLEAN SALA

In [16]:
classes['Sala'] = classes['Sala'].str.replace(
    r"(lab\.).+",
    r"\1",
    regex=True
)

SyntaxError: incomplete input (1032503299.py, line 4)

###  EXPORT

In [ ]:
if not os.path.exists("schedules"):
    os.makedirs("schedules")
classes.to_csv(f"schedules/{group}.csv")

classes